# Phrase Files Formatter (Magic Box)

_last updated: 2026/04/23_

- Colab now installs the project from GitHub instead of embedding copied source code
- Processing runs through the same shared Python entrypoint as the local CLI

## Instructions

1. Run **Step 1: Install From GitHub** to install the project and its dependencies into Colab.
2. Run **Step 2: Upload Files** and choose one or more `.docx` and `.mxliff` pairs with matching base names.
3. Run **Step 3: Process Files** to call the shared pipeline from the installed repo.
4. Run **Step 4: Download Results** if you want Colab to download every merged output file automatically.
5. Each merged file is saved as `<base>_merged.docx`.

_Note: If prompted by your browser, allow multiple automatic downloads. Some browsers block this by default; see the manual for an example screenshot._

—

Tip: Use `Ctrl` (or `Cmd` on Mac) to select multiple files.


In [ ]:
# @title Step 1: Install From GitHub {display-mode: "form"}
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "pip",
    "setuptools",
    "wheel",
])

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--verbose",
    "git+https://github.com/dariru3/py-process_phrase_files.git@refactor-cli-colab-unification",
])


In [ ]:
# @title Step 2: Upload Files {display-mode: "form"}
import os
from google.colab import files
import shutil

INPUT_FOLDER = '/content/MagicBox/'
OUTPUT_FOLDER = '/content/MagicBox/Output_Folder/'

def ensure_folder_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

def clear_folder(folder_path):
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as exc:
                print(f'Error: failed to clear {file_path}. Reason: {exc}')

def upload_files(upload_path):
    clear_folder(upload_path)
    ensure_folder_exists(upload_path)

    print("\nUpload one or more .docx and .mxliff files.")
    print("Pairs are matched by the same base filename.\n")
    uploaded = files.upload()

    if not uploaded:
        print("No files uploaded.")
        return False

    docx_map = {os.path.splitext(fn)[0]: fn for fn in uploaded.keys() if fn.lower().endswith('.docx')}
    mxliff_map = {os.path.splitext(fn)[0]: fn for fn in uploaded.keys() if fn.lower().endswith('.mxliff')}
    paired_basenames = sorted(set(docx_map.keys()) & set(mxliff_map.keys()))

    if len(paired_basenames) == 0:
        print("Error: No valid pairs found. Ensure each .docx has a matching .mxliff with the same base name.")
        for fn in uploaded.keys():
            try:
                os.remove(fn)
            except FileNotFoundError:
                pass
        return False

    print(f"Found {len(paired_basenames)} pair(s):")
    for base in paired_basenames:
        docx_file = docx_map[base]
        mxliff_file = mxliff_map[base]
        print(f"- {docx_file}  <->  {mxliff_file}")
        shutil.move(docx_file, os.path.join(upload_path, docx_file))
        shutil.move(mxliff_file, os.path.join(upload_path, mxliff_file))

    paired_files = set(docx_map[b] for b in paired_basenames) | set(mxliff_map[b] for b in paired_basenames)
    for fn in uploaded.keys():
        if fn not in paired_files:
            try:
                os.remove(fn)
            except FileNotFoundError:
                pass

    return True

ensure_folder_exists(INPUT_FOLDER)
ensure_folder_exists(OUTPUT_FOLDER)

if upload_files(INPUT_FOLDER):
    print("\nFile upload successful.")
else:
    print("\nFile upload failed. Please follow the instructions and try again.")


In [ ]:
# @title Step 3: Process Files {display-mode: "form"}
from src.pipeline import run_pipeline

INPUT_FOLDER = "/content/MagicBox/"
OUTPUT_FOLDER = "/content/MagicBox/Output_Folder/"
FORCE_REPROCESS = False  # @param {type:"boolean"}

processed_count = run_pipeline(
    INPUT_FOLDER,
    OUTPUT_FOLDER,
    skip_existing=not FORCE_REPROCESS,
)
print(f"Processed {processed_count} file pair(s).")


In [ ]:
# @title Step 4: Download Results {display-mode: "form"}
import os
from google.colab import files

OUTPUT_FOLDER = "/content/MagicBox/Output_Folder/"

downloaded = 0
for filename in sorted(os.listdir(OUTPUT_FOLDER)):
    file_path = os.path.join(OUTPUT_FOLDER, filename)
    if os.path.isfile(file_path):
        downloaded += 1
        files.download(file_path)

print(f"Started download for {downloaded} file(s).")
